# Complete SPARC Validation Test

Runs the fixed boundary relation, baryons-only baseline, RAR/MOND benchmark, 10,000-shuffle control, and per-galaxy NFW benchmark with AIC and BIC.

**Use:** Runtime → Run all.


In [ ]:
!pip -q install numpy pandas scipy matplotlib


In [ ]:
import os, shutil, glob, json, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from IPython.display import display, Markdown

repo_dir = '/content/Galaxy-Rotation-Curves-SPARC-Validation-Test'
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
!git clone -q https://github.com/SPruynIDR/Galaxy-Rotation-Curves-SPARC-Validation-Test.git "$repo_dir"
os.chdir(repo_dir)

catalog_candidates = glob.glob('*SPARC_Lelli2016c*.txt')
mass_candidates = glob.glob('*MassModels_Lelli2016c*.txt')
if not catalog_candidates:
    raise FileNotFoundError('No SPARC catalog file found in repository.')
if not mass_candidates:
    raise FileNotFoundError('No SPARC mass-model file found in repository.')
catalog_file = sorted(catalog_candidates)[0]
mass_file = sorted(mass_candidates)[0]
print('Catalog:', catalog_file)
print('Mass models:', mass_file)


In [ ]:
def parse_catalog(path):
    rows = []
    with open(path, encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if i < 98:
                continue
            p = line.split()
            if len(p) < 18:
                continue
            try:
                rows.append({'Galaxy': p[0], 'T': int(p[1]), 'L36': float(p[7]), 'MHI': float(p[13]), 'RHI': float(p[14]), 'Qflag': int(p[17])})
            except Exception:
                pass
    return pd.DataFrame(rows).drop_duplicates('Galaxy')

def parse_mass_models(path):
    rows = []
    with open(path, encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if i < 25:
                continue
            p = line.split()
            if len(p) < 10:
                continue
            try:
                rows.append({'Galaxy': p[0], 'D': float(p[1]), 'R': float(p[2]), 'Vobs': float(p[3]), 'eVobs': float(p[4]), 'Vgas': float(p[5]), 'Vdisk': float(p[6]), 'Vbul': float(p[7])})
            except Exception:
                pass
    return pd.DataFrame(rows)

def rms(y, yhat):
    y = np.asarray(y); yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y-yhat)**2)))

def r2(y, yhat):
    y = np.asarray(y); yhat = np.asarray(yhat)
    return float(1 - np.sum((y-yhat)**2)/np.sum((y-y.mean())**2))

catalog = parse_catalog(catalog_file)
mass = parse_mass_models(mass_file)
eligible = catalog[catalog['RHI'] > 0].copy()
data = mass.merge(eligible, on='Galaxy', how='inner')
data['Vbar2'] = data['Vdisk']**2 + data['Vbul']**2 + data['Vgas']*np.abs(data['Vgas'])
excluded = data[data['Vbar2'] < 0][['Galaxy','R','Vbar2']].copy()
data = data[data['Vbar2'] >= 0].copy()
data['Vbar'] = np.sqrt(data['Vbar2'])
data['Mbar'] = (0.5*data['L36'] + 1.33*data['MHI']) * 1e9

A0 = 10**(-0.473464); q = 0.25; nprof = 0.8513; x0 = 0.4466
data['x'] = data['R']/data['RHI']
data['F'] = data['x']**nprof/(data['x']**nprof + x0**nprof)
data['Vbnd'] = A0*data['Mbar']**q*data['F']
data['Vpred_boundary'] = np.sqrt(data['Vbar2'] + data['Vbnd']**2)
obs = data['Vobs'].to_numpy()
boundary = data['Vpred_boundary'].to_numpy()
baryons = data['Vbar'].to_numpy()
print('Galaxies:', data['Galaxy'].nunique(), 'Points:', len(data))


In [ ]:
# RAR/MOND benchmark
KPC_M = 3.085677581491367e19
gdagger = 1.2e-10
gbar = (data['Vbar'].to_numpy()*1000.0)**2/(data['R'].to_numpy()*KPC_M)
grar = gbar/(1.0-np.exp(-np.sqrt(gbar/gdagger)))
vrar = np.sqrt(grar*data['R'].to_numpy()*KPC_M)/1000.0

base_results = {
    'Boundary relation': {'RMS': rms(obs,boundary), 'R2': r2(obs,boundary)},
    'RAR/MOND': {'RMS': rms(obs,vrar), 'R2': r2(obs,vrar)},
    'Baryons only': {'RMS': rms(obs,baryons), 'R2': r2(obs,baryons)},
}
base_results


In [ ]:
# 10,000-shuffle control
rng = np.random.default_rng(20260710)
galaxies = eligible['Galaxy'].to_numpy()
rhi_values = eligible['RHI'].to_numpy()
mbar_map = ((0.5*eligible.set_index('Galaxy')['L36'] + 1.33*eligible.set_index('Galaxy')['MHI'])*1e9)
base = data[['Galaxy','R','Vobs','Vbar2']].copy()
true_rms = base_results['Boundary relation']['RMS']
shuffle_rms = np.empty(10000)
for j in range(10000):
    rhi_map = dict(zip(galaxies, rng.permutation(rhi_values)))
    rhi = base['Galaxy'].map(rhi_map).to_numpy()
    mbar = base['Galaxy'].map(mbar_map).to_numpy()
    xx = base['R'].to_numpy()/rhi
    ff = xx**nprof/(xx**nprof + x0**nprof)
    vb = A0*mbar**q*ff
    pred = np.sqrt(base['Vbar2'].to_numpy()+vb**2)
    shuffle_rms[j] = rms(base['Vobs'].to_numpy(),pred)
shuffle_summary = {
    'true_rms': true_rms,
    'best_shuffle_rms': float(shuffle_rms.min()),
    'n_better': int(np.sum(shuffle_rms < true_rms)),
    'n_shuffle': 10000,
    'seed': 20260710,
}
shuffle_summary


In [ ]:
# NFW benchmark
G = 4.30091e-6
lower = np.array([4.0,-2.0]); upper = np.array([12.0,3.0])
starts = [(a,b) for a in [5.5,7.5,9.5,11.0] for b in [-1.0,0.0,1.0,2.0]]
def nfw_v2(r, logrho, logrs):
    rho = 10**logrho; rs = 10**logrs
    xx = np.maximum(r/rs,1e-12)
    menc = 4*np.pi*rho*rs**3*(np.log1p(xx)-xx/(1+xx))
    return G*menc/np.maximum(r,1e-12)

fit_rows = []; pred_parts = []
for galaxy,g in data.groupby('Galaxy',sort=True):
    rr = g['R'].to_numpy(float); yy = g['Vobs'].to_numpy(float); vb2 = g['Vbar2'].to_numpy(float)
    def pred(theta):
        return np.sqrt(np.maximum(vb2+nfw_v2(rr,theta[0],theta[1]),0.0))
    best_rss = np.inf; best_theta = None
    for start in starts:
        res = least_squares(lambda t: pred(t)-yy, x0=np.asarray(start,float), bounds=(lower,upper), method='trf', xtol=1e-10, ftol=1e-10, gtol=1e-10, max_nfev=2000)
        rss = float(np.sum(res.fun**2))
        if rss < best_rss:
            best_rss = rss; best_theta = res.x.copy()
    out = g.copy(); out['Vpred_NFW'] = pred(best_theta); pred_parts.append(out)
    fit_rows.append({'Galaxy':galaxy,'log10_rho_s':float(best_theta[0]),'log10_r_s':float(best_theta[1]),'rss':best_rss})
nfw_pred = pd.concat(pred_parts,ignore_index=True)
fits = pd.DataFrame(fit_rows)
nfw_obs = nfw_pred['Vobs'].to_numpy(); nfw_model = nfw_pred['Vpred_NFW'].to_numpy()
nfw_rms = rms(nfw_obs,nfw_model); nfw_r2 = r2(nfw_obs,nfw_model)
nfw_rms, nfw_r2


In [ ]:
# Final model comparison
N = len(data)
rss_boundary = float(np.sum((obs-boundary)**2))
rss_nfw = float(np.sum((nfw_obs-nfw_model)**2))
aic_boundary = N*np.log(rss_boundary/N)+2*3
bic_boundary = N*np.log(rss_boundary/N)+3*np.log(N)
aic_nfw = N*np.log(rss_nfw/N)+2*342
bic_nfw = N*np.log(rss_nfw/N)+342*np.log(N)

summary = pd.DataFrame([
    ['Boundary relation',base_results['Boundary relation']['RMS'],base_results['Boundary relation']['R2'],aic_boundary,bic_boundary],
    ['RAR/MOND',base_results['RAR/MOND']['RMS'],base_results['RAR/MOND']['R2'],np.nan,np.nan],
    ['Baryons only',base_results['Baryons only']['RMS'],base_results['Baryons only']['R2'],np.nan,np.nan],
    ['NFW halo',nfw_rms,nfw_r2,aic_nfw,bic_nfw],
],columns=['Model','RMS_km_s','R2','AIC','BIC'])

display(Markdown('## COMPLETE RESULT'))
display(summary)
display(Markdown(f"**Shuffle control:** {shuffle_summary['n_better']} of 10,000 beat the true assignment. Best shuffled RMS = {shuffle_summary['best_shuffle_rms']:.10f} km/s."))
display(Markdown(f"**ΔAIC (NFW − boundary):** {aic_nfw-aic_boundary:.4f}  \n**ΔBIC (NFW − boundary):** {bic_nfw-bic_boundary:.4f}  \nAIC favors NFW. BIC favors the three-parameter global boundary relation."))

summary.to_csv('complete_model_summary.csv',index=False)
fits.to_csv('complete_nfw_per_galaxy_fits.csv',index=False)
with open('complete_shuffle_summary.json','w') as f:
    json.dump(shuffle_summary,f,indent=2)
